## What is each station's rated capacity, and does it reconcile with the paper?

In [3]:
import pandas as pd
import numpy as np 
import json 
from pathlib import Path

In [10]:
ROOT = Path().cwd().parent / "data" 

In [12]:
RAW = ROOT / "raw" 

In [13]:
df = pd.read_csv(RAW / "devices.csv")

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   device_hash_id   66 non-null     str    
 1   source           66 non-null     int64  
 2   station_hash_id  66 non-null     str    
 3   device_id        66 non-null     str    
 4   name             53 non-null     str    
 5   device_model     65 non-null     str    
 6   max_power        53 non-null     float64
dtypes: float64(1), int64(1), str(5)
memory usage: 3.7 KB


In [15]:
df.head()

,device_hash_id,source,station_hash_id,device_id,name,device_model,max_power
0,3223f6ee747ede5f21c5858a9ddc6fce,2,e2afd8af435a86d8bfe72dcbe937a5d6,REDACTED,50KTL-M3(COM3-13),SUN2000-50KTL-M3,55.0
1,15de11e7df8168a60db8f55669d66e76,2,b59685487a1fc5712e03c264f70cb656,REDACTED,INV-1,SUN2000-50KTL-M3,55.0
2,a2bc58707e3ac5b41a4a8f49900d675b,1,2c85da17f3ac504327da04874ef1a701,REDACTED,NaN,1603,100.0
3,44aaeb88850bacfd88cce3f4fc07566f,2,e2afd8af435a86d8bfe72dcbe937a5d6,REDACTED,50KTL-M3(COM3-19),SUN2000-50KTL-M3,55.0
4,94aedd81bad035cc73968fe5b06f808a,2,19477f81a2058c850a09310b7795fd4f,REDACTED,INV-6T2289011404,SUN2000-40KTL-M3,44.0


In [16]:
print(df[df["max_power"].isna()][["device_model", "name", "station_hash_id"]])

    device_model                    name                   station_hash_id
5   Smart Logger                Logger-1  55cddba5799e63f231eae0eb8cca8247
13           NaN                 Meter-1  19477f81a2058c850a09310b7795fd4f
18  Smart Logger                Logger-1  7534daaee746a48bc5b99132810f3e69
26    PowerMeter  Meter-AM00102275571727  e2afd8af435a86d8bfe72dcbe937a5d6
29    PowerMeter                 Meter-1  b59685487a1fc5712e03c264f70cb656
30    PowerMeter                 Meter-2  b59685487a1fc5712e03c264f70cb656
32   SDongleA-05     Dongle-BT2260684709  ae68bd65b1c87d7a5985d1d5498442af
33  Smart Logger     Logger-102275571727  e2afd8af435a86d8bfe72dcbe937a5d6
35    PowerMeter                 Meter-1  55cddba5799e63f231eae0eb8cca8247
41   SDongleA-05     Dongle-BT22B0734239  19477f81a2058c850a09310b7795fd4f
42    PowerMeter                 Meter-1  7534daaee746a48bc5b99132810f3e69
47  Smart Logger     Logger-102275905588  b59685487a1fc5712e03c264f70cb656
65    PowerMeter  Meter-A

In [18]:
print(df[df["device_model"].isna()])

                      device_hash_id  source  \
13  c84b78eddb67e5ad02e16137af8a3f74       2   

                     station_hash_id device_id     name device_model  \
13  19477f81a2058c850a09310b7795fd4f  REDACTED  Meter-1          NaN   

    max_power  
13        NaN  


In [24]:
assert df["max_power"].notna().sum() == 53, "inverter count drifted from 53"

In [38]:
fact_inv = pd.read_csv(RAW / "hourly_pv_weather_inverter.csv", usecols=["device_hash_id"])
producing = set(fact_inv["device_hash_id"].unique())

In [39]:
inverter = df[df["max_power"].notna()].copy()

In [40]:
inverter.head()

,device_hash_id,source,station_hash_id,device_id,name,device_model,max_power
0,3223f6ee747ede5f21c5858a9ddc6fce,2,e2afd8af435a86d8bfe72dcbe937a5d6,REDACTED,50KTL-M3(COM3-13),SUN2000-50KTL-M3,55.0
1,15de11e7df8168a60db8f55669d66e76,2,b59685487a1fc5712e03c264f70cb656,REDACTED,INV-1,SUN2000-50KTL-M3,55.0
2,a2bc58707e3ac5b41a4a8f49900d675b,1,2c85da17f3ac504327da04874ef1a701,REDACTED,NaN,1603,100.0
3,44aaeb88850bacfd88cce3f4fc07566f,2,e2afd8af435a86d8bfe72dcbe937a5d6,REDACTED,50KTL-M3(COM3-19),SUN2000-50KTL-M3,55.0
4,94aedd81bad035cc73968fe5b06f808a,2,19477f81a2058c850a09310b7795fd4f,REDACTED,INV-6T2289011404,SUN2000-40KTL-M3,44.0


In [41]:
inverter["produced"] = inverter["device_hash_id"].isin(producing)

In [42]:
inverter["produced"].head()

0    True
1    True
2    True
3    True
4    True
Name: produced, dtype: bool

In [43]:
inverter["produced"].tail()

60    True
61    True
62    True
63    True
64    True
Name: produced, dtype: bool

In [44]:
type(inverter)

pandas.DataFrame

In [47]:
print("inverters with max_power:", len(inverter))
print("of which producing:", inverter["produced"].sum())
print(inverter[~inverter["produced"]][["station_hash_id", "device_model", "max_power"]])

inverters with max_power: 53
of which producing: 50
                     station_hash_id      device_model  max_power
11  b59685487a1fc5712e03c264f70cb656  SUN2000-50KTL-M3       55.0
28  b59685487a1fc5712e03c264f70cb656  SUN2000-50KTL-M3       55.0
40  b59685487a1fc5712e03c264f70cb656  SUN2000-50KTL-M3       55.0


## Summing the capacity of each station and assign the label based on its capacity

In [53]:
cap = (inverter[inverter["produced"]]
    .groupby(["station_hash_id"])["max_power"]
    .sum()
    .rename("capacity_kw")
    .sort_values())

print(cap)

station_hash_id
19477f81a2058c850a09310b7795fd4f      44.0
ae68bd65b1c87d7a5985d1d5498442af      44.0
55cddba5799e63f231eae0eb8cca8247     220.0
7534daaee746a48bc5b99132810f3e69     440.0
b59685487a1fc5712e03c264f70cb656     605.0
e2afd8af435a86d8bfe72dcbe937a5d6     605.0
2c85da17f3ac504327da04874ef1a701    1300.0
Name: capacity_kw, dtype: float64


In [54]:
print(pd.cut(cap, [0, 100, 500, float("inf")],
             labels=["small", "medium", "large"]).value_counts())

capacity_kw
large     3
small     2
medium    2
Name: count, dtype: int64


In [56]:
import json

ordered = cap.sort_values()   # ascending: S1 = smallest
labels = {
    h: {"label": f"S{i+1}", "capacity_kw": float(kw)}
    for i, (h, kw) in enumerate(ordered.items())
}

# hash stays the key; label is presentation only — same discipline as
# never letting a UI string become a DB key
out = ROOT / "processed" / "station_labels.json"
out.write_text(json.dumps(labels, indent=2))
print(json.dumps(labels, indent=2))

{
  "19477f81a2058c850a09310b7795fd4f": {
    "label": "S1",
    "capacity_kw": 44.0
  },
  "ae68bd65b1c87d7a5985d1d5498442af": {
    "label": "S2",
    "capacity_kw": 44.0
  },
  "55cddba5799e63f231eae0eb8cca8247": {
    "label": "S3",
    "capacity_kw": 220.0
  },
  "7534daaee746a48bc5b99132810f3e69": {
    "label": "S4",
    "capacity_kw": 440.0
  },
  "b59685487a1fc5712e03c264f70cb656": {
    "label": "S5",
    "capacity_kw": 605.0
  },
  "e2afd8af435a86d8bfe72dcbe937a5d6": {
    "label": "S6",
    "capacity_kw": 605.0
  },
  "2c85da17f3ac504327da04874ef1a701": {
    "label": "S7",
    "capacity_kw": 1300.0
  }
}
